In [0]:
from pyspark.sql import functions as F

# Carregando as tabelas da camada Bronze para o Spark e convertendo a data corretamente
df_orders = spark.table("bronze_olist_orders_dataset").withColumn(
    "order_purchase_timestamp", 
    F.to_timestamp("order_purchase_timestamp", "yyyy-MM-dd HH:mm:ss")
)
df_reviews = spark.table("bronze_olist_order_reviews_dataset")
df_items = spark.table("bronze_olist_order_items_dataset")
df_customers = spark.table("bronze_olist_customers_dataset")

In [0]:
# 1. Tratamento e preparação da tabela de Reviews
# Foi criada a coluna binária: 1 se review_score >= 4, senão 0
df_reviews_prepared = df_reviews.select(
    "order_id",
    "review_score",
    F.when(F.col("review_score") >= 4, 1).otherwise(0).alias("target_high_review")
).dropDuplicates(["order_id"]) # Evita duplicação caso haja mais de um review por pedido

# 2. Agregação dos itens do pedido 
df_items_agg = df_items.groupBy("order_id").agg(
    F.sum("price").alias("total_price"),
    F.sum("freight_value").alias("total_freight"),
    F.count("order_item_id").alias("item_count")
)

# 3. Cruzamento das tabelas para formar a base analítica consolidada
df_silver = df_orders.join(df_customers, "customer_id", "inner") \
                     .join(df_items_agg, "order_id", "inner") \
                     .join(df_reviews_prepared, "order_id", "inner")

In [0]:
# Selecionando colunas relevantes para o negócio e para o modelo preditivo
df_gold = df_silver.select(
    "order_id",
    "customer_id",
    "customer_state",
    "customer_city",
    "order_status",
    "order_purchase_timestamp",
    "total_price",
    "total_freight",
    "item_count",
    "review_score"
).withColumn(
    "target_high_review",
    F.when(F.expr("try_cast(review_score as int)") >= 4, 1).otherwise(0)
)

# Visualizando uma amostra da base Gold pronta para modelagem
display(df_gold.limit(10))

# Salvando a tabela Gold finalizada
df_gold.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_olist_model_ready")